# 10 — Predicción de ticket medio, largo plazo (sin memoria reciente)

Mismo planteamiento que el notebook 09, aplicado al ticket medio: un modelo alternativo sin ninguna variable de memoria (`ticket_medio_*`, `facturacion_*`, `num_tickets_*`, `ticket_mediano_1d_antes`), pensado para fechas demasiado lejanas para tener un historial reciente real. Se espera un error notablemente mayor que el modelo de corto plazo (`07_prediccion_ticket_medio_definitivo.ipynb`, MAE test 13.85 €) — es el coste honesto de quitar la señal más fuerte, no un fallo del modelo.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ml.features import build_ticket_feature_table, TICKET_FEATURE_COLS

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

RANDOM_STATE = 42
MODELS_DIR = PROJECT_ROOT / 'results' / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'ticket_medio_largo_plazo'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Datos: mismas filas que el notebook 07, sin las columnas de memoria

In [2]:
MEMORIA_COLS = [c for c in TICKET_FEATURE_COLS if c.startswith(('ticket_medio_', 'facturacion_', 'num_tickets_')) or c == 'ticket_mediano_1d_antes']
FEATURE_COLS = [c for c in TICKET_FEATURE_COLS if c not in MEMORIA_COLS]
print(f'Columnas de memoria descartadas ({len(MEMORIA_COLS)}): {MEMORIA_COLS}')
print(f'\nColumnas del modelo de largo plazo ({len(FEATURE_COLS)}): {FEATURE_COLS}')

df = build_ticket_feature_table()
X = df[FEATURE_COLS].copy()
y = df['ticket_medio'].copy()
fechas = df['fecha'].copy()
print(f'\nFilas disponibles: {len(X)}')

Columnas de memoria descartadas (22): ['ticket_medio_1d_antes', 'ticket_medio_3d_antes', 'ticket_medio_7d_antes', 'ticket_medio_14d_antes', 'ticket_medio_21d_antes', 'ticket_medio_28d_antes', 'ticket_medio_media_3d', 'ticket_medio_media_7d', 'ticket_medio_media_14d', 'ticket_medio_media_28d', 'ticket_medio_std_7d', 'ticket_medio_std_28d', 'ticket_medio_tendencia_7_28', 'facturacion_1d_antes', 'facturacion_7d_antes', 'facturacion_14d_antes', 'facturacion_media_7d', 'num_tickets_1d_antes', 'num_tickets_7d_antes', 'num_tickets_14d_antes', 'num_tickets_media_7d', 'ticket_mediano_1d_antes']

Columnas del modelo de largo plazo (39): ['es_festivo', 'festivo_nombre', 'tiene_evento', 'intensidad_evento', 'impacto_evento', 'direccion_evento', 'categoria_evento', 'cat_evento', 'temperature_max', 'temperature_min', 'temperature_mean', 'precipitation_mm', 'precipitation_hours', 'wind_speed_max', 'sunshine_duration_h', 'reservas_anticipadas', 'comensales_anticipados', 'grupos_grandes_anticipados', '


Filas disponibles: 242


## 2. Split train/test cronológico (80/20)

In [3]:
split = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split].copy(), X.iloc[split:].copy()
y_train, y_test = y.iloc[:split].copy(), y.iloc[split:].copy()
fechas_train, fechas_test = fechas.iloc[:split], fechas.iloc[split:]
print('TRAIN:', fechas_train.min().date(), '->', fechas_train.max().date(), f'({len(X_train)} días)')
print('TEST :', fechas_test.min().date(), '->', fechas_test.max().date(), f'({len(X_test)} días)')

TRAIN: 2025-10-02 -> 2026-05-13 (193 días)
TEST : 2026-05-14 -> 2026-07-09 (49 días)


## 3. Modelos: Ridge (ganador en el notebook 07) y Random Forest

In [4]:
categorical_cols = X_train.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()
numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()

categorical_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
numeric_linear = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
numeric_tree = Pipeline([('imputer', SimpleImputer(strategy='median'))])

preprocessor_linear = ColumnTransformer([('num', numeric_linear, numeric_cols), ('cat', categorical_pipe, categorical_cols)])
preprocessor_tree = ColumnTransformer([('num', numeric_tree, numeric_cols), ('cat', categorical_pipe, categorical_cols)])

tscv = TimeSeriesSplit(n_splits=5)

pipe_ridge = Pipeline([('preprocessor', preprocessor_linear), ('model', Ridge(random_state=RANDOM_STATE))])
search_ridge = RandomizedSearchCV(pipe_ridge, {'model__alpha': [0.1, 0.3, 1, 3, 10, 30, 100, 300]}, n_iter=8, cv=tscv, scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1)
search_ridge.fit(X_train, y_train)

pipe_rf = Pipeline([('preprocessor', preprocessor_tree), ('model', RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=1))])
search_rf = RandomizedSearchCV(
    pipe_rf,
    {'model__n_estimators': [200, 400], 'model__max_depth': [None, 4, 6, 10], 'model__min_samples_leaf': [1, 2, 4], 'model__max_features': [0.5, 0.7, 1.0]},
    n_iter=10, cv=tscv, scoring='neg_mean_absolute_error', random_state=RANDOM_STATE, n_jobs=-1,
)
search_rf.fit(X_train, y_train)

print(f'Ridge  MAE_CV = {-search_ridge.best_score_:.2f} €  params: {search_ridge.best_params_}')
print(f'RF     MAE_CV = {-search_rf.best_score_:.2f} €  params: {search_rf.best_params_}')

Ridge  MAE_CV = 18.39 €  params: {'model__alpha': 30}
RF     MAE_CV = 17.61 €  params: {'model__n_estimators': 200, 'model__min_samples_leaf': 1, 'model__max_features': 0.5, 'model__max_depth': 10}


## 4. Evaluación en test — comparación honesta con baseline y con el modelo de corto plazo

In [5]:
def smape(y_true, y_pred):
    denom = np.abs(y_true) + np.abs(y_pred)
    m = denom != 0
    return np.mean(2 * np.abs(y_pred[m] - y_true[m]) / denom[m]) * 100

candidatos = {'Ridge (largo plazo)': search_ridge.best_estimator_, 'Random Forest (largo plazo)': search_rf.best_estimator_}
resultados = []
for nombre, modelo in candidatos.items():
    pred = modelo.predict(X_test)
    resultados.append({'modelo': nombre, 'MAE_€': mean_absolute_error(y_test, pred), 'RMSE_€': mean_squared_error(y_test, pred) ** 0.5, 'R2': r2_score(y_test, pred), 'sMAPE_%': smape(y_test.values, pred)})

baseline_naive = y.shift(1).iloc[split:].values
mask_b = ~np.isnan(baseline_naive)
resultados.append({'modelo': 'Baseline (día anterior)', 'MAE_€': mean_absolute_error(y_test[mask_b], baseline_naive[mask_b]), 'RMSE_€': mean_squared_error(y_test[mask_b], baseline_naive[mask_b]) ** 0.5, 'R2': r2_score(y_test[mask_b], baseline_naive[mask_b]), 'sMAPE_%': smape(y_test[mask_b].values, baseline_naive[mask_b])})

MAE_CORTO_PLAZO = 13.85  # notebook 07, Ridge, ya evaluado alli
resultados.append({'modelo': 'Modelo corto plazo (referencia, notebook 07)', 'MAE_€': MAE_CORTO_PLAZO, 'RMSE_€': None, 'R2': None, 'sMAPE_%': None})

comparacion = pd.DataFrame(resultados).sort_values('MAE_€')
display(comparacion)

mejor_nombre = min(candidatos, key=lambda n: mean_absolute_error(y_test, candidatos[n].predict(X_test)))
mejor_modelo = candidatos[mejor_nombre]
print(f'\nMejor modelo de largo plazo: {mejor_nombre}')

,modelo,MAE_€,RMSE_€,R2,sMAPE_%
3,"Modelo corto plazo (referencia, notebook 07)",13.850000,NaN,NaN,NaN
0,Ridge (largo plazo),14.874271,17.689782,0.499517,15.645639
1,Random Forest (largo plazo),17.067851,19.083723,0.417534,17.790080
2,Baseline (día anterior),23.039477,30.493771,-0.487189,23.813499



Mejor modelo de largo plazo: Ridge (largo plazo)


## 5. Conclusión

El ticket medio ya tenía, en el modelo de corto plazo, un techo de precisión bajo por su "ruido de composición" (sección 17 del notebook 07). Sin memoria reciente ese techo debería subir todavía más — hay que comprobarlo en la tabla anterior antes de dar el modelo por válido para uso real.

## 6. Guardado del modelo

In [6]:
ruta_modelo = MODELS_DIR / 'mejor_modelo_ticket_medio_largo_plazo.joblib'
joblib.dump(mejor_modelo, ruta_modelo)
comparacion.to_csv(RESULTS_DIR / 'comparacion_ticket.csv', index=False)
print(f'Modelo guardado en: {ruta_modelo}')

Modelo guardado en: C:\Users\CandelaGB\Desktop\TFM anita\TFM-Hosteleria-AI\results\models\mejor_modelo_ticket_medio_largo_plazo.joblib
